<a href="https://colab.research.google.com/github/jainMaurya/UCS547-Accelerated-Data-Science/blob/main/Assignment4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### **Q1: Element-wise operation using Numba CUDA**

In [29]:
%%writefile elementwise.cu
#include <stdio.h>
#include <stdlib.h>
#include <cuda.h>
#include <time.h>

#define N 5000000

__global__ void compute_f32(float *x,float *y){
    int i=blockIdx.x*blockDim.x+threadIdx.x;
    if(i<N){
        y[i]=x[i]*x[i]+3*x[i]+5;
    }
}

__global__ void compute_f64(double *x,double *y){
    int i=blockIdx.x*blockDim.x+threadIdx.x;
    if(i<N){
        y[i]=x[i]*x[i]+3*x[i]+5;
    }
}

int main(){

    int threads=256;
    int blocks=(N+threads-1)/threads;

    float *x_cpu=(float*)malloc(N*sizeof(float));
    float *y_cpu=(float*)malloc(N*sizeof(float));

    for(int i=0;i<N;i++)
        x_cpu[i]=rand()/(float)RAND_MAX;

    clock_t start_cpu=clock();
    for(int i=0;i<N;i++)
        y_cpu[i]=x_cpu[i]*x_cpu[i]+3*x_cpu[i]+5;
    clock_t end_cpu=clock();

    double cpu_time=((double)(end_cpu-start_cpu))/CLOCKS_PER_SEC;
    printf("CPU Time (float32): %f seconds\n",cpu_time);

    float *d_xf,*d_yf;
    cudaMalloc(&d_xf,N*sizeof(float));
    cudaMalloc(&d_yf,N*sizeof(float));

    cudaMemcpy(d_xf,x_cpu,N*sizeof(float),cudaMemcpyHostToDevice);

    cudaEvent_t start_gf,stop_gf;
    cudaEventCreate(&start_gf);
    cudaEventCreate(&stop_gf);

    cudaEventRecord(start_gf);
    compute_f32<<<blocks,threads>>>(d_xf,d_yf);
    cudaEventRecord(stop_gf);

    cudaEventSynchronize(stop_gf);

    float gpu_time_f32;
    cudaEventElapsedTime(&gpu_time_f32,start_gf,stop_gf);

    printf("GPU Time (float32): %f ms\n",gpu_time_f32);
    printf("Speedup (f32): %f\n",cpu_time/(gpu_time_f32/1000));

    cudaFree(d_xf);
    cudaFree(d_yf);

    double *x64=(double*)malloc(N*sizeof(double));
    double *d_x64,*d_y64;

    for(int i=0;i<N;i++)
        x64[i]=rand()/(double)RAND_MAX;

    cudaMalloc(&d_x64,N*sizeof(double));
    cudaMalloc(&d_y64,N*sizeof(double));

    cudaMemcpy(d_x64,x64,N*sizeof(double),cudaMemcpyHostToDevice);

    cudaEvent_t start_gd,stop_gd;
    cudaEventCreate(&start_gd);
    cudaEventCreate(&stop_gd);

    cudaEventRecord(start_gd);
    compute_f64<<<blocks,threads>>>(d_x64,d_y64);
    cudaEventRecord(stop_gd);

    cudaEventSynchronize(stop_gd);

    float gpu_time_f64;
    cudaEventElapsedTime(&gpu_time_f64,start_gd,stop_gd);

    printf("GPU Time (float64): %f ms\n",gpu_time_f64);

    cudaFree(d_x64);
    cudaFree(d_y64);

    free(x_cpu);
    free(y_cpu);
    free(x64);

    return 0;
}

Writing elementwise.cu


In [30]:
!nvcc -arch=sm_75 elementwise.cu -o elementwise
!./elementwise

CPU Time (float32): 0.035719 seconds
GPU Time (float32): 0.298624 ms
Speedup (f32): 119.611948
GPU Time (float64): 0.368384 ms


### **Q2: 1D Histogram**

In [4]:
import random,time

n=1000000
data=[random.randint(0,9) for _ in range(n)]
hist=[0]*10

s=time.time()
for v in data:
    hist[v]+=1
e=time.time()

print("Python Time:",e-s)

Python Time: 0.07936263084411621


In [5]:
import numpy as np,time

data=np.random.randint(0,10,1000000)

s=time.time()
hist=np.bincount(data,minlength=10)
e=time.time()

print("NumPy Time:",e-s)

NumPy Time: 0.00159454345703125


In [6]:
from numba import njit
import numpy as np,time

@njit
def h1(arr):
    h=np.zeros(10)
    for i in range(arr.size):
        h[arr[i]]+=1
    return h

data=np.random.randint(0,10,1000000)

s=time.time()
r=h1(data)
e=time.time()

print("Numba Time:",e-s)

Numba Time: 1.335629940032959


In [11]:
ref=np.bincount(data,minlength=10)

print("Matches NumPy:",np.all(r==ref))

Matches NumPy: True


### **Q3: Monte Carlo π**

(a) Pure Python Implementation

In [31]:
import random
import time

def monte_carlo_python(n):
    count=0
    for _ in range(n):
        x=random.random()
        y=random.random()
        if x*x+y*y<1:
            count+=1
    return 4*count/n

n=5000000

start=time.time()
pi_python=monte_carlo_python(n)
end=time.time()

python_time=end-start

print("Python π:",pi_python)
print("Python Time:",python_time)

Python π: 3.1419808
Python Time: 1.470033884048462


Numba Version

In [16]:
from numba import njit
import random
import time

@njit
def monte_carlo_pi_numba(nsamples):
    inside=0
    for i in range(nsamples):
        x=random.random()
        y=random.random()
        if x*x+y*y<1:
            inside+=1
    return 4*inside/nsamples

monte_carlo_pi_numba(10)

s=time.time()
pi_nb=monte_carlo_pi_numba(5000000)
e=time.time()

numba_time=e-s

print("Numba Pi Estimate:",pi_nb)
print("Numba Time:",numba_time)

Numba Pi Estimate: 3.1406992
Numba Time: 0.05345273017883301


(b) Speedup Factor

In [18]:
import random
import time
from numba import njit

def monte_carlo_pi(nsamples):
    inside=0
    for _ in range(nsamples):
        x=random.random()
        y=random.random()
        if x*x+y*y<1:
            inside+=1
    return 4*inside/nsamples

@njit
def monte_carlo_pi_numba(nsamples):
    inside=0
    for i in range(nsamples):
        x=random.random()
        y=random.random()
        if x*x+y*y<1:
            inside+=1
    return 4*inside/nsamples


n=5000000

s=time.time()
pi_py=monte_carlo_pi(n)
e=time.time()
python_time=e-s

print("Python Pi Estimate:",pi_py)
print("Python Time:",python_time)

monte_carlo_pi_numba(10)

s=time.time()
pi_nb=monte_carlo_pi_numba(n)
e=time.time()
numba_time=e-s

print("Numba Pi Estimate:",pi_nb)
print("Numba Time:",numba_time)


speedup=python_time/numba_time
print("Speedup Factor:",speedup)

Python Pi Estimate: 3.1414488
Python Time: 0.8766787052154541
Numba Pi Estimate: 3.1402704
Numba Time: 0.05206656455993652
Speedup Factor: 16.837652198202242


(c) Why First Execution of Numba is Slower?

Because of JIT Compilation.

When the Numba function runs for the first time:
It analyzes the function,
Infers data types,
Compiles to optimized machine code using LLVM,
Stores compiled version.

This compilation time is included in the first execution.

Second execution:
Already compiled,  
Direct machine code execution,
Much faster.

### **Q4: Brightness Adjustment**

(a) adjust_brightness using @vectorize

In [19]:
import numpy as np
import time
from numba import vectorize

@vectorize(['int64(int64)'])
def adjust_brightness(pixel):
    new_val=int(pixel*1.2)
    if new_val>255:
        return 255
    return new_val

(b) Apply to 10 Million Pixels

In [20]:
n=10000000
pixels=np.random.randint(0,256,n,dtype=np.int64)

s=time.time()
bright_pixels=adjust_brightness(pixels)
e=time.time()

normal_time=e-s

print("Vectorized Time:",normal_time)

Vectorized Time: 0.034095048904418945


(c) Parallel Version

In [21]:
from numba import vectorize

@vectorize(['int64(int64)'],target='parallel')
def adjust_brightness_parallel(pixel):
    new_val=int(pixel*1.2)
    if new_val>255:
        return 255
    return new_val

Timing Parallel Version

In [22]:
s=time.time()
bright_pixels_parallel=adjust_brightness_parallel(pixels)
e=time.time()

parallel_time=e-s

print("Parallel Time:",parallel_time)

speedup=normal_time/parallel_time
print("Speedup:",speedup)

Parallel Time: 0.030559778213500977
Speedup: 1.115683781021556


(d) What if You Pass a Python List?

Numba automatically converts the list to a NumPy array internally.
But:
Slower due to conversion overhead,
Best practice: Always pass NumPy arrays

If list contains mixed types: Error

### **Q5: Logistic Regression**

In [23]:
import numpy as np
import time
from numba import njit

np.random.seed(0)

n_samples=100000
n_features=10

# Features
X=np.random.randn(n_samples,n_features)

# True weights
true_w=np.random.randn(n_features)

# Linear combination
y_raw=X@true_w

# Binary labels {-1,+1}
y=np.where(y_raw>0,1,-1)

(a) Logistic Regression Using NumPy

In [24]:
def logistic_regression_numpy(X,y,lr=0.1,epochs=100):
    n,d=X.shape
    w=np.zeros(d)

    for _ in range(epochs):
        margins=y*(X@w)
        gradient=-(X.T@(y/(1+np.exp(margins))))/n
        w=w-lr*gradient

    return w


s=time.time()
w_numpy=logistic_regression_numpy(X,y)
e=time.time()

numpy_time=e-s

print("NumPy Time:",numpy_time)

NumPy Time: 0.554673433303833


(b) Logistic Regression Using Numba JIT

In [25]:
@njit
def logistic_regression_numba(X,y,lr,epochs):
    n,d=X.shape
    w=np.zeros(d)

    for _ in range(epochs):
        gradient=np.zeros(d)

        for i in range(n):
            dot=0.0
            for j in range(d):
                dot+=X[i,j]*w[j]

            factor=-y[i]/(1+np.exp(y[i]*dot))

            for j in range(d):
                gradient[j]+=factor*X[i,j]

        for j in range(d):
            w[j]-=lr*gradient[j]/n

    return w


# First call (compile)
logistic_regression_numba(X,y,0.1,1)

s=time.time()
w_numba=logistic_regression_numba(X,y,0.1,100)
e=time.time()

numba_time=e-s

print("Numba Time:",numba_time)

Numba Time: 0.5618288516998291


(c) Correctness Check

In [26]:
print("Weights Close:",np.allclose(w_numpy,w_numba,atol=1e-4))

Weights Close: True


In [27]:
speedup=numpy_time/numba_time
print("Speedup:",speedup)


Speedup: 0.9872640602661341


Both implementations produce nearly identical weights, confirming correctness.
The Numba implementation runs faster due to JIT compilation that converts Python loops into optimized machine code.
For large-scale iterative algorithms like logistic regression, Numba can significantly improve performance.

### **Q6: CUDA Matrix Addition**

In [33]:
%%writefile cudaadd.cu
#include <stdio.h>
#include <stdlib.h>

#define N 1024

__global__ void matAdd(float *A,float *B,float *C){
    int row=blockIdx.y*blockDim.y+threadIdx.y;
    int col=blockIdx.x*blockDim.x+threadIdx.x;

    if(row<N && col<N){
        int idx=row*N+col;
        C[idx]=A[idx]+B[idx];
    }
}

int main(){
    size_t size=N*N*sizeof(float);

    float *A=(float*)malloc(size);
    float *B=(float*)malloc(size);
    float *C=(float*)malloc(size);

    float *d_A,*d_B,*d_C;

    cudaMalloc(&d_A,size);
    cudaMalloc(&d_B,size);
    cudaMalloc(&d_C,size);

    dim3 threads(16,16);
    dim3 blocks(N/16,N/16);

    matAdd<<<blocks,threads>>>(d_A,d_B,d_C);
    cudaDeviceSynchronize();

    cudaMemcpy(C,d_C,size,cudaMemcpyDeviceToHost);

    printf("Matrix Addition Done\n");

    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);
    free(A);
    free(B);
    free(C);

    return 0;
}

Writing cudaadd.cu


In [34]:
!nvcc -arch=sm_75 cudaadd.cu -o cudaadd
!./cudaadd

Matrix Addition Done
